# 3.4 · 缓存与订阅：让列表保持新鲜

**Caching and subscriptions: keeping lists fresh**

> **学完本课你能：** 设置并遵守 ttlMs/cacheScope 提示，构建正确的客户端缓存，并使用 subscriptions/listen 接收列表变更和资源更新通知。

官方阅读材料：[Spec: Caching](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/caching) · [Spec: Subscriptions](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/subscriptions)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 每个可缓存结果都携带提示
默认情况下，`MCPServer` 会把结果标记为 `ttlMs: 0`（立即过期）和 `cacheScope: "private"`——这是最安全的选择。检查每一个可缓存方法，包括客户端在连接时获取的 `server/discover` 结果，并确认 `tools/call` 不携带提示。

In [ ]:
import json
import logging
from collections import Counter

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)

plain = MCPServer("plain")


@plain.tool()
def ping_tool() -> str:
    """Return pong."""
    return "pong"


@plain.resource("config://app")
def config() -> str:
    return "theme=dark"


@plain.prompt()
def hello() -> str:
    """Say hello."""
    return "Hello!"


async with Client(plain) as client:
    discover = client.session.discover_result
    rows = [("server/discover", discover), ("tools/list", await client.list_tools()),
            ("prompts/list", await client.list_prompts()), ("resources/list", await client.list_resources()),
            ("resources/templates/list", await client.list_resource_templates()),
            ("resources/read", await client.read_resource("config://app")),
            ("tools/call", await client.call_tool("ping_tool", {}))]
    for method, result in rows:
        wire = result.model_dump(by_alias=True, mode="json", exclude_none=True)
        print(f"{method:<26} ttlMs={wire.get('ttlMs', '—')!s:<7} cacheScope={wire.get('cacheScope', '—')}")

### 2. 服务器端：选择提示
`MCPServer(cache_hints={method: CacheHint(ttl_ms, scope)})` 按方法设置提示。一个中间件统计实际到达服务器（Server）的请求数，这样我们就能看到客户端缓存在起作用。

In [ ]:
from mcp.server.caching import CacheHint

hits = Counter()


async def count_requests(ctx, call_next):
    hits[ctx.method] += 1
    return await call_next(ctx)


server = MCPServer("travel", middleware=[count_requests], cache_hints={
    "tools/list": CacheHint(ttl_ms=300_000, scope="public"),       # same for every user, 5 min
    "resources/read": CacheHint(ttl_ms=60_000, scope="private"),    # user data, 1 min
})
inbox = {"messages": 1}


@server.tool()
def search_flights(origin: str, destination: str) -> list[str]:
    """Search flights between two airports."""
    return [f"{origin}->{destination} 08:15", f"{origin}->{destination} 17:40"]


@server.resource("inbox://me", mime_type="application/json")
def my_inbox() -> str:
    return json.dumps(inbox)


async with Client(server) as client:
    t = await client.list_tools()
    r = await client.read_resource("inbox://me")
    print("tools/list    ", t.ttl_ms, t.cache_scope)
    print("resources/read", r.ttl_ms, r.cache_scope)

### 3. 客户端：响应缓存与 `cache_mode`
SDK 的 `Client` 默认遵守提示（`cache=CacheConfig()`）。每个可缓存调用都接受 `cache_mode`：`"use"`（新鲜则从缓存提供，否则获取并存储）、`"refresh"`（总是获取，然后存储）、`"bypass"`（忽略缓存）。`Client(..., cache=None)` 会禁用缓存。

In [ ]:
async with Client(server) as client:
    hits.clear()
    await client.list_tools()                         # miss -> server
    await client.list_tools()                         # fresh -> served from cache
    await client.list_tools(cache_mode="refresh")     # forced fetch, re-stored
    await client.list_tools(cache_mode="bypass")      # straight to server
    print("with cache:   ", dict(hits))

async with Client(server, cache=None) as client:
    hits.clear()
    await client.list_tools()
    await client.list_tools()
    print("cache=None:   ", {k: v for k, v in hits.items() if k != "server/discover"})

### 4. 新鲜度从接收时间开始计算
`CacheConfig(clock=...)` 让我们可以伪造挂钟时间：当 `now < t_received + ttlMs` 时条目是新鲜的，之后就过期（Stale）。

In [ ]:
from mcp.client.caching import CacheConfig

fake_now = [1_000.0]  # seconds

async with Client(server, cache=CacheConfig(clock=lambda: fake_now[0])) as client:
    hits.clear()
    for advance_s in [0, 120, 179.9, 0.2]:
        fake_now[0] += advance_s
        before = hits["tools/list"]
        await client.list_tools()
        print(f"t+{fake_now[0] - 1_000:6.1f}s  ->  {'server' if hits['tools/list'] > before else 'cache '}"
              f"   (ttl {300}s)")

在 t=0 收到的条目，在 120 s 和 299.9 s 时是新鲜的（`now < 0 + 300 s`），在 300.1 s 时过期，所以那次请求会发往服务器，新的响应开始一轮新的 300 s 计时。新鲜度是<strong>惰性</strong>检查的，在需要数据的那一刻进行：计时期间没有任何东西在后台被重新获取。

### 5. TTL 是提示而非保证——通知会使缓存失效
一个工具修改了收件箱。没有订阅（Subscription）时，客户端会一直提供缓存的（此时已错误的）值，直到 TTL 用完。有了 `subscriptions/listen`，`notifications/resources/updated` 会立即逐出该条目，下一次读取就会发往服务器。

In [ ]:
@server.tool()
async def receive_mail(ctx: Context) -> int:
    """Simulate new mail arriving."""
    inbox["messages"] += 1
    await ctx.notify_resource_updated("inbox://me")
    return inbox["messages"]


async with Client(server) as client:
    print("read:", (await client.read_resource("inbox://me")).contents[0].text)
    await client.call_tool("receive_mail", {})
    print("after change, no subscription:", (await client.read_resource("inbox://me")).contents[0].text, "<- stale!")

    async with client.listen(resource_subscriptions=["inbox://me"]) as sub:
        await client.call_tool("receive_mail", {})
        event = await anext(sub)
        hits.clear()
        print("event:", event)
        print("after notification:", (await client.read_resource("inbox://me")).contents[0].text,
              "| reached server:", hits["resources/read"] == 1)

### 6. 线路上的 `subscriptions/listen`
深入 SDK 之下，通过 stdio：两个过滤器不同的<strong>并发</strong>订阅、最先到达的确认、`subscriptionId` 标记、选择加入式过滤、通过 `notifications/cancelled` 进行的客户端取消，以及由服务器发起的<strong>优雅关闭（Graceful Closure）</strong>。（`shutdown_streams` 工具调用了 SDK 的一个内部接口，只是为了触发服务器在关机时会做的事情。）

In [ ]:
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
import time
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "watch_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    from mcp.server.caching import CacheHint
    from mcp.server.mcpserver import Context

    server = MCPServer("watchable", cache_hints={"tools/list": CacheHint(ttl_ms=300_000, scope="public")})
    config = {"theme": "dark"}

    @server.resource("config://app", mime_type="application/json")
    def app_config() -> str:
        return str(config)

    @server.tool()
    async def set_theme(theme: str, ctx: Context) -> str:
        """Change the theme (updates config://app)."""
        config["theme"] = theme
        await ctx.notify_resource_updated("config://app")
        return "ok"

    @server.tool()
    async def install_plugin(name: str, ctx: Context) -> str:
        """Install a plugin, which adds a tool."""
        server.add_tool(lambda: f"{name} says hi", name=f"{name}_hello", description=f"Greeting from {name}")
        await ctx.notify_tools_changed()
        return "installed"

    @server.tool()
    async def shutdown_streams() -> str:
        """Demo only: end all listen streams gracefully, as on server shutdown."""
        server._lowlevel_server._request_handlers["subscriptions/listen"].handler.close()
        return "closing"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "watch_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
inbound: queue.Queue = queue.Queue()
threading.Thread(target=lambda: [inbound.put(line) for line in proc.stdout], daemon=True).start()
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def send(message):
    message.setdefault("params", {})["_meta"] = META
    shown = {k: v for k, v in message.items() if k != "jsonrpc"}
    shown["params"] = {k: v for k, v in message["params"].items() if k != "_meta"}
    print(">>>", json.dumps(shown))
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


def drain(seconds=0.6):
    deadline = time.time() + seconds
    while (left := deadline - time.time()) > 0:
        try:
            msg = json.loads(inbound.get(timeout=left))
        except queue.Empty:
            break
        if "result" in msg and "tools" in msg["result"]:
            msg["result"] = {k: msg["result"][k] for k in ("ttlMs", "cacheScope")} | {"tools": "[...]"}
        elif "result" in msg:
            msg["result"].get("_meta", {}).pop("io.modelcontextprotocol/serverInfo", None)
            msg["result"] = {k: v for k, v in msg["result"].items()
                             if k in ("resultType", "structuredContent") or (k == "_meta" and v)}
        print("    <<<", json.dumps({k: v for k, v in msg.items() if k != "jsonrpc"}))


send({"jsonrpc": "2.0", "id": 1, "method": "tools/list"}); drain()
send({"jsonrpc": "2.0", "id": "tools-sub", "method": "subscriptions/listen",
      "params": {"notifications": {"toolsListChanged": True}}})
send({"jsonrpc": "2.0", "id": "cfg-sub", "method": "subscriptions/listen",
      "params": {"notifications": {"resourceSubscriptions": ["config://app"]}}}); drain()
print("\n# a resource change reaches only cfg-sub:")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "set_theme", "arguments": {"theme": "light"}}}); drain()
print("\n# a tool-list change reaches only tools-sub:")
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "install_plugin", "arguments": {"name": "maps"}}}); drain()
print("\n# cancel cfg-sub, then change the resource again: silence")
send({"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "cfg-sub"}}); drain(0.3)
send({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "set_theme", "arguments": {"theme": "solarized"}}}); drain()
print("\n# the server shuts its streams down gracefully:")
send({"jsonrpc": "2.0", "id": 5, "method": "tools/call", "params": {"name": "shutdown_streams", "arguments": {}}}); drain()
proc.stdin.close()
_ = proc.wait(timeout=10)

仔细阅读：每条确认都先于其订阅上的任何通知；每条流消息都把 listen 请求的 id 作为 `subscriptionId` 携带；最后一条消息是 id 为 `tools-sub` 的<strong>响应</strong>——也就是优雅关闭。

### 7. 用 SDK 做同样的事：`client.listen()`
`client.listen(...)` 发送请求，等待确认（`sub.honored`、`sub.subscription_id`），然后产出带类型的事件。两个订阅可以同时运行；当服务器优雅关闭（Graceful Closure）时，`async for` 循环就会直接结束。

In [ ]:
catalogue = MCPServer("catalogue")
prompts_enabled = []


@catalogue.tool()
async def add_feature(name: str, ctx: Context) -> str:
    """Add a prompt and a tool, then tell subscribers."""
    catalogue.add_tool(lambda: name, name=f"{name}_tool", description=f"{name} tool")
    await ctx.notify_tools_changed()
    prompts_enabled.append(name)
    await ctx.notify_prompts_changed()
    return "added"


async with Client(catalogue) as client:
    async with client.listen(tools_list_changed=True) as tools_sub, \
               client.listen(prompts_list_changed=True, resources_list_changed=True) as other_sub:
        print("subscription ids:", tools_sub.subscription_id, other_sub.subscription_id)
        print("honoured:", tools_sub.honored.model_dump(by_alias=True, exclude_none=True),
              other_sub.honored.model_dump(by_alias=True, exclude_none=True))
        await client.call_tool("add_feature", {"name": "maps"})
        print(f"{tools_sub.subscription_id} got:", await anext(tools_sub))
        print(f"{other_sub.subscription_id} got:", await anext(other_sub))

        # graceful closure from the server side (SDK internal, demo only)
        catalogue._lowlevel_server._request_handlers["subscriptions/listen"].handler.close()
        leftovers = [event async for event in tools_sub]
        print("tools_sub ended gracefully; leftover events:", leftovers)

### 8. 动手破坏：共享缓存中被错标为 `public` 的结果
一个网关在用户之间共享同一个缓存存储，按授权上下文分区，并选择启用对 `public` 条目的共享（`share_public=True`）。如果服务器把一次<em>私有</em>的收件箱读取标为 `public`，Bob 就会拿到 Alice 的邮件，而服务器甚至从未看到他的请求。

In [ ]:
from mcp.client.caching import InMemoryResponseCacheStore

mail = {"alice": "Alice: your flight is confirmed", "bob": "Bob: nothing new"}


def make_mail_server(scope):
    srv = MCPServer("mail", middleware=[count_requests],
                    cache_hints={"resources/read": CacheHint(ttl_ms=60_000, scope=scope)})
    current_user = {"name": "alice"}   # stand-in for the authenticated principal

    @srv.resource("mail://inbox")
    def mailbox() -> str:
        return mail[current_user["name"]]

    return srv, current_user


for scope in ["public", "private"]:
    srv, current_user = make_mail_server(scope)
    store = InMemoryResponseCacheStore()   # one shared gateway cache
    hits.clear()
    for user in ["alice", "bob"]:
        current_user["name"] = user
        config = CacheConfig(store=store, partition=user, target_id="mail-server", share_public=True)
        async with Client(srv, cache=config) as client:
            text = (await client.read_resource("mail://inbox")).contents[0].text
            print(f"cacheScope={scope:<7} {user:<5} sees: {text!r}")
    print(f"   server saw {hits['resources/read']} resources/read request(s)\n")

使用 `private` 时，每个用户的条目都位于自己的分区中，Bob 的读取会发往服务器。修复应当在<strong>服务器</strong>端进行（把用户数据标为 `private`，并对每个请求实施访问控制）——而网关只应对其信任的服务器启用 `share_public`。

### 动手练习
1. 为 `server` 上的 `prompts/list` 设置 10 分钟的 public TTL，并用中间件计数器验证第二次 `list_prompts()` 不会到达服务器。然后通过一个工具用 `ctx.notify_prompts_changed()` 添加一个提示模板，并展示只有当客户端以 `prompts_list_changed=True` 进行监听时，缓存才会失效（Invalidation）。
2. 在第 6 节中，打开第三个只请求 `promptsListChanged` 的订阅，再次运行 `install_plugin`，确认它什么也收不到。是哪条规范规则保证了这一点？
3. 在第 6 节中，发送一个 `notifications` 为空对象的 `subscriptions/listen`。确认消息里包含什么？一个谨慎的客户端会如何处理它？
4. 为宿主编写一个 `fetch_tools(client)` 辅助函数：列表新鲜时提供缓存的列表，在任何 `ToolsListChanged` 事件之后用 `cache_mode="refresh"` 重新获取，如果重新获取抛出异常则退回到过期的副本。